In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# ==============================================================================
# 2. MAPEAMENTO DE CONDUTAS (O QUE GERA O RETORNO)
# ==============================================================================
# Vamos varrer o texto de conduta/evolução em busca de motivos de 'Não Resolvido'
keywords = {
    'Solicitação de Exames': ['exame', 'laborat', 'sangue', 'rx', 'raio-x', 'ultrassom', 'solicito'],
    'Encaminhamento Presencial': ['presencial', 'ubs', 'upa', 'exame fisico', 'palpação'],
    'Encaminhamento Especialista': ['especialista', 'encaminho', 'parecer', 'cardiologista'],
    'Retorno Agendado': ['retorno', 'voltar', 'dias', 'reavaliar']
}

# Busca as colunas de texto médico (Conduta/Evolução)
cols_texto = [c for c in df.columns if any(p in c.lower() for p in ['conduta', 'evolucao', 'plano', 'obs'])]

resultados = {}
for categoria, termos in keywords.items():
    count = 0
    for col in cols_texto:
        count += df[col].astype(str).str.contains('|'.join(termos), case=False, na=False).sum()
    resultados[categoria] = count

resumo = pd.Series(resultados).sort_values(ascending=False)

# ==============================================================================
# 3. VISUALIZAÇÃO GRÁFICA
# ==============================================================================
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(x=resumo.values, y=resumo.index, palette="flare")
plt.title("Motivos de Não Resolução no Primeiro Contato (N=52.160)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Número de Menções (Frequência de Conduta)", fontsize=12)
plt.ylabel("")

# Adiciona os valores diretamente na frente de cada barra (para facilitar a leitura)
for container in ax.containers:
    ax.bar_label(container, padding=4, fontweight='bold', color='#333333', fontsize=11)

# Limpeza visual (remove as bordas do topo e da direita)
sns.despine(right=True, top=True)

plt.tight_layout()
plt.show()

# ==============================================================================
# 4. EXPORTAÇÃO SILENCIOSA DA TABELA
# ==============================================================================
# Converte a Series gerada em um DataFrame bem formatado para o Excel
df_tabela = resumo.reset_index()
df_tabela.columns = ['Motivo de Não Resolução', 'Frequência de Menções']

# Adiciona uma coluna de porcentagem para enriquecer o relatório
total_mencoes = df_tabela['Frequência de Menções'].sum()
if total_mencoes > 0:
    df_tabela['Proporção (%)'] = ((df_tabela['Frequência de Menções'] / total_mencoes) * 100).round(1)
else:
    df_tabela['Proporção (%)'] = 0.0

nome_arquivo = 'Tabela_Motivos_Nao_Resolucao.csv'
df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

# ==============================================================================
# 5. DIAGNÓSTICO EXECUTIVO NO TERMINAL
# ==============================================================================
print("\n" + "="*80)
print("DIAGNÓSTICO EXECUTIVO: LIMITADORES DE RESOLUTIVIDADE")
print("="*80)
if not resumo.empty and resumo.iloc[0] > 0:
    print(f"O maior impedimento de resolução imediata é: {resumo.index[0]} (n={resumo.iloc[0]})")
else:
    print("Nenhuma menção encontrada para as categorias pesquisadas.")
print("-" * 80)
print(f"📂 Tabela exportada silenciosamente com sucesso em: '{nome_arquivo}'")
print("="*80 + "\n")

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO E SANEAMENTO
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except:
    print("Erro ao ler o arquivo.")
    raise

# Filtro Base: Apenas o que foi atendimento (Record ID presente)
df_ev = df.dropna(subset=['Record ID']).copy()

# ==============================================================================
# 2. CLASSIFICAÇÃO PELO FUNIL DE DECISÃO (O que gera os seus 7k)
# ==============================================================================
col_texto = next((c for c in df_ev.columns if any(p in c.lower() for p in ['conduta', 'plano', 'evolucao'])), None)

def classificar_motivo(texto):
    t = str(texto).lower()
    if len(t) < 5 or 'não compareceu' in t: return 'Outros'
    
    # Ordem de prioridade para bater com seu gráfico de motivos
    if any(k in t for k in ['presencial', 'exame fisico', 'upa']): return 'Encaminhamento Presencial'
    if any(k in t for k in ['especialista', 'encaminho', 'parecer']): return 'Encaminhamento Especialista'
    if any(k in t for k in ['exame', 'laborat', 'sangue', 'rx']): return 'Solicitação de Exames'
    if any(k in t for k in ['retorno', 'reavaliar', 'agendar', 'voltar']): return 'Retorno Agendado'
    
    return 'Resolvido/Alta'

df_ev['Desfecho'] = df_ev[col_texto].apply(classificar_motivo) if col_texto else 'Outros'

# ==============================================================================
# 3. EXTRAÇÃO DOS NÚMEROS REAIS (CÁLCULO DINÂMICO)
# ==============================================================================
n_total_base = len(df_ev)

# Valores das 4 categorias do seu gráfico
v_retorno = df_ev[df_ev['Desfecho'] == 'Retorno Agendado'].shape[0]
v_exames = df_ev[df_ev['Desfecho'] == 'Solicitação de Exames'].shape[0]
v_especialista = df_ev[df_ev['Desfecho'] == 'Encaminhamento Especialista'].shape[0]
v_presencial = df_ev[df_ev['Desfecho'] == 'Encaminhamento Presencial'].shape[0]

# Total Não Resolvido (Soma exata das barras)
n_nao_resolvido = v_retorno + v_exames + v_especialista + v_presencial

# O que sobra é Resolvido
n_resolvido = n_total_base - n_nao_resolvido

# ==============================================================================
# 4. SANKEY DIAGRAM (EQUILÍBRIO DE MASSA)
# ==============================================================================
# Nós: 0:Total, 1:Resolvido, 2:Não Resolvido, 3:Retorno, 4:Exames, 5:Especialista, 6:Presencial
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 20, thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = [
          f"TOTAL ATENDIMENTOS<br>(N={n_total_base})",         # 0
          f"RESOLVIDO / ALTA<br>({n_resolvido})",             # 1
          f"NÃO RESOLVIDO<br>({n_nao_resolvido})",            # 2
          f"RETORNO AGENDADO<br>({v_retorno})",                # 3
          f"SOLICITAÇÃO DE EXAMES<br>({v_exames})",            # 4
          f"ENCAMINHAMENTO ESPECIALISTA<br>({v_especialista})",# 5
          f"ENCAMINHAMENTO PRESENCIAL<br>({v_presencial})"     # 6
      ],
      color = ["#2c3e50", "#27ae60", "#e67e22", "#D99C88", "#C3777C", "#9A596E", "#623D5B"]
    ),
    link = dict(
      source = [0, 0, 2, 2, 2, 2], 
      target = [1, 2, 3, 4, 5, 6],
      value = [n_resolvido, n_nao_resolvido, v_retorno, v_exames, v_especialista, v_presencial],
      color = "rgba(200, 200, 200, 0.3)"
    )
)])

fig.update_layout(title_text="<b>Fluxo de Resolutividade TeleSAP (Dados Sincronizados)</b>", font_size=12)
fig.show()

# Conferência no Terminal para o seu Artigo
print(f"\n--- VALIDAÇÃO DOS DADOS ---")
print(f"Total Base (N): {n_total_base}")
print(f"Não Resolvidos (Soma): {n_nao_resolvido}")
print(f"Resolvidos (N - Não Resolvidos): {n_resolvido}")
print(f"Balanço: {n_resolvido + n_nao_resolvido} (Deve ser {n_total_base})")

In [ ]:
import pandas as pd

# ==============================================================================
# 1. CARREGAMENTO E SANEAMENTO DOS DADOS (A BASE)
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
df = pd.read_csv(file_path, low_memory=False)

# Definimos o N TOTAL como todos os atendimentos que possuem prontuário (Record ID)
df_base = df.dropna(subset=['Record ID']).copy()
n_total = len(df_base)

# ==============================================================================
# 2. DEFINIÇÃO DA LÓGICA DE FLUXO (O MOTOR DO DIAGRAMA)
# ==============================================================================
col_texto = next((c for c in df_base.columns if any(p in c.lower() for p in ['conduta', 'plano', 'evolucao'])), None)

def mapear_fluxo(texto):
    t = str(texto).lower()
    if len(t) < 5: return 'Resolvido/Alta'
    
    # Hierarquia para bater com sua tabela de 7.794 retornos
    if any(k in t for k in ['presencial', 'exame fisico', 'upa']): return 'Encaminhamento Presencial'
    if any(k in t for k in ['especialista', 'encaminho', 'parecer']): return 'Encaminhamento Especialista'
    if any(k in t for k in ['exame', 'laborat', 'sangue', 'rx']): return 'Solicitação de Exames'
    if any(k in t for k in ['retorno', 'reavaliar', 'agendar', 'voltar']): return 'Retorno Agendado'
    
    return 'Resolvido/Alta'

df_base['Desfecho'] = df_base[col_texto].apply(mapear_fluxo) if col_texto else 'Resolvido/Alta'

# ==============================================================================
# 3. CONSTRUÇÃO DA TABELA DE NODOS (A BASE DO SANKEY)
# ==============================================================================
# Contamos as ocorrências reais
resumo = df_base['Desfecho'].value_counts()

v_retorno = resumo.get('Retorno Agendado', 0)
v_exames = resumo.get('Solicitação de Exames', 0)
v_especialista = resumo.get('Encaminhamento Especialista', 0)
v_presencial = resumo.get('Encaminhamento Presencial', 0)
v_nao_resolvido = v_retorno + v_exames + v_especialista + v_presencial
v_resolvido = n_total - v_nao_resolvido

# Criamos a estrutura "Source-Target-Value"
base_sankey = pd.DataFrame([
    # Nível 1: Total -> Grandes Grupos
    {'Source': 'Total Atendimentos', 'Target': 'Resolvido/Alta', 'Value': v_resolvido},
    {'Source': 'Total Atendimentos', 'Target': 'Não Resolvido', 'Value': v_nao_resolvido},
    
    # Nível 2: Não Resolvido -> Motivos Específicos
    {'Source': 'Não Resolvido', 'Target': 'Retorno Agendado', 'Value': v_retorno},
    {'Source': 'Não Resolvido', 'Target': 'Solicitação de Exames', 'Value': v_exames},
    {'Source': 'Não Resolvido', 'Target': 'Encaminhamento Especialista', 'Value': v_especialista},
    {'Source': 'Não Resolvido', 'Target': 'Encaminhamento Presencial', 'Value': v_presencial}
])

# ==============================================================================
# 4. VALIDAÇÃO FINAL
# ==============================================================================
print("-" * 50)
print(f"BASE DO DIAGRAMA GERADA (N={n_total})")
print("-" * 50)
print(base_sankey)
print("-" * 50)
print(f"Soma das saídas: {v_resolvido + v_nao_resolvido} (Confere com Total: {'SIM' if v_resolvido+v_nao_resolvido == n_total else 'NÃO'})")

# Exporta para você usar em qualquer ferramenta de gráfico
base_sankey.to_csv('base_diagrama_sankey.csv', index=False)